# Shapes

This example creates the shapes of Geom4hep, calls the shape functions on them and draws them.
The shapes have the parameters of the corresponding Geant4 solids.

## Loading the modules
`CairoMakie` draws into images; use `GLMakie` instead for interactive windows. Makie exports
names that are also shapes of Geom4hep (e.g. `Box`), so only the names that are needed are
imported from it.

In [ ]:
using Geom4hep, Geom4hep.Units
using LinearAlgebra
using CairoMakie: CairoMakie, Figure, LScene

const T = Float64;

## Creating shapes
The constructors take the floating point type as parameter, and the parameters of the shape
with the conventions of the Geant4 solids: half lengths, radii, and angular ranges given by a start
angle and an extent. Lengths are in mm and angles in radians; the units of `Geom4hep.Units` make
this explicit.

In [ ]:
box    = Box{T}(10cm, 15cm, 20cm)
tube   = Tube{T}(5cm, 10cm, 20cm, 0, 270deg)
cone   = Cone{T}(2cm, 5cm, 5cm, 10cm, 20cm, 0, 2π)
trd    = Trd{T}(10cm, 5cm, 15cm, 8cm, 20cm)
trap   = Trap{T}(20cm, 10deg, 30deg, 10cm, 8cm, 12cm, 5deg, 6cm, 5cm, 7cm, 5deg)
sphere = Sphere{T}(8cm, 10cm, 0, 270deg, 30deg, 120deg)
torus  = Torus{T}(2cm, 4cm, 12cm, 0, 270deg)
pcone  = Polycone{T}([0, 5cm, 5cm, 0], [8cm, 10cm, 10cm, 6cm], [-20cm, -10cm, 10cm, 20cm], 0, 2π)
phedra = Polyhedra{T}(6, [3cm, 3cm, 3cm], [6cm, 10cm, 6cm], [-20cm, 0, 20cm], 0, 2π)
ctube  = CutTube{T}(5cm, 10cm, 20cm, 0, 2π, Vector3{T}(0, -0.5, -1), Vector3{T}(0.5, 0, 1));

Printing a shape shows its parameters

In [ ]:
torus

## Drawing shapes
With a Makie backend loaded, `draw(shape)` creates a figure with the shape, and `draw!` adds it to
a scene. Here all the shapes are drawn in a grid.

In [ ]:
shapes = [box, tube, cone, trd, trap, sphere, torus, pcone, phedra, ctube]
fig = Figure(size=(1000, 450))
for (i, s) in enumerate(shapes)
    scene = LScene(fig[(i-1) ÷ 5 + 1, (i-1) % 5 + 1], show_axis=false)
    draw!(scene, s)
end
fig

## Shape functions
The shape functions take points and directions in the frame of the shape.
`inside` classifies a point as `kInside`, `kSurface` or `kOutside`:

In [ ]:
p_in, p_out = Point3{T}(0, 0, 0), Point3{T}(30cm, 0, 0)
inside(box, p_in) == kInside, inside(box, p_out) == kOutside, inside(box, Point3{T}(10cm, 0, 0)) == kSurface

`distance_to_in` is the distance along a direction from an outside point to the shape, and
`distance_to_out` from an inside point to the surface:

In [ ]:
dir = Vector3{T}(-1, 0, 0)
distance_to_in(box, p_out, dir), distance_to_out(box, p_in, dir)

A ray that misses the shape gets `Inf`:

In [ ]:
distance_to_in(box, p_out, Vector3{T}(0, 1, 0))

The safeties are lower bounds of the distance to the surface in any direction:

In [ ]:
safety_to_in(box, p_out), safety_to_out(box, Point3{T}(5cm, 0, 0))

`normal` gives the outward normal at a point of the surface, `extent` the corners of a bounding
box, and `capacity` and `surface` the volume and the area:

In [ ]:
normal(sphere, Point3{T}(0, 10cm, 0))

In [ ]:
extent(torus)

In [ ]:
capacity(box) ≈ 8 * 10cm * 15cm * 20cm, surface(box)

## Checking the distances
`draw_distance_to_in` shoots random rays from outside a shape and draws the points where they
enter it; `draw_distance_to_out` does the same from inside. Both draw the surface of the shape,
which is a good visual test of the distance functions.

In [ ]:
draw_distance_to_out(phedra, 50_000)

## Boolean shapes
Boolean shapes combine two shapes; the second one is placed in the frame of the first one with a
`Transformation3D`. They can be nested.

In [ ]:
hole  = Tube{T}(0, 4cm, 30cm, 0, 2π)
frame = BooleanSubtraction(box, hole)
frame = BooleanSubtraction(frame, hole, Transformation3D{T}(0, 0, 0, π/2, 0, 0))
frame = BooleanUnion(frame, Sphere{T}(0, 6cm), Transformation3D{T}(0, 0, 20cm))
inside(frame, Point3{T}(0, 0, 0)) == kOutside, inside(frame, Point3{T}(0, 0, 24cm)) == kInside

The rays shot by `draw_distance_to_in` show the result:

In [ ]:
draw_distance_to_in(frame, 50_000)

## Single precision
All the shapes can also be created in single precision, which is what most GPUs need. The
surfaces are then thicker (`tolerance(Float32)` is 1 µm) to absorb the rounding errors.

In [ ]:
box32 = Box{Float32}(10cm, 15cm, 20cm)
distance_to_in(box32, Point3{Float32}(30cm, 0, 0), Vector3{Float32}(-1, 0, 0)), tolerance(Float32)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*